In [4]:
import logging
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Configure Logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# 1. Load Data & Prepare
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")
df["date_time"] = pd.to_datetime(df["date_time"])
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Quartile Binning for Congestion[cite: 1]
q1, q2, q3 = df["traffic_volume"].quantile([0.25, 0.5, 0.75]).values


def bucket(v):
    if v <= q1:
        return "Low"
    elif v <= q2:
        return "Medium"
    elif v <= q3:
        return "High"
    return "Severe"


df["congestion_category"] = df["traffic_volume"].apply(bucket)

# 2. K-Means Clustering[cite: 1]
weather_severity_map = {
    "Clear": 0,
    "Clouds": 1,
    "Drizzle": 2,
    "Mist": 3,
    "Rain": 4,
    "Fog": 5,
    "Haze": 5,
    "Snow": 6,
    "Thunderstorm": 7,
    "Squall": 8,
}
df["weather_severity"] = (
    df["weather_main"].map(weather_severity_map).fillna(1)
)

cluster_features = ["hour", "weather_severity", "traffic_volume", "temp"]
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(df[cluster_features])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_cluster_scaled)

cluster_summary = df.groupby("cluster")[cluster_features].mean()
cluster_summary["count"] = df["cluster"].value_counts()
cluster_summary["dominant_congestion"] = df.groupby("cluster")[
    "congestion_category"
].agg(lambda x: x.mode()[0])

print("\n" + "=" * 65)
print("K-MEANS CLUSTERING SUMMARY & PROFILES")
print("=" * 65)
print(cluster_summary.round(2).to_string())


# 3. Native Association Analysis (Pure Pandas/Numpy)[cite: 1]
def time_of_day(hour):
    if 6 <= hour < 10:
        return "Morning_Peak"
    elif 10 <= hour < 16:
        return "Midday"
    elif 16 <= hour < 20:
        return "Evening_Peak"
    return "OffPeak_Night"


df["time_of_day"] = df["hour"].apply(time_of_day)
df["day_type"] = df["is_weekend"].map({0: "Weekday", 1: "Weekend"})

# Group-based rule evaluation
total_records = len(df)
rules_list = []

targets = ["Low", "Medium", "High", "Severe"]
group_cols = ["time_of_day", "day_type", "weather_main"]

for keys, group in df.groupby(group_cols):
    group_size = len(group)
    if group_size < 100:  # Minimum support filter
        continue

    for target in targets:
        target_count = (group["congestion_category"] == target).sum()
        if target_count == 0:
            continue

        support = target_count / total_records
        confidence = target_count / group_size
        baseline_prob = (df["congestion_category"] == target).mean()
        lift = confidence / baseline_prob

        if confidence >= 0.3 and lift > 1.1:
            rules_list.append(
                {
                    "Antecedents": f"{keys[0]}, {keys[1]}, Weather:{keys[2]}",
                    "Consequent (Congestion)": target,
                    "Support": round(support, 4),
                    "Confidence": round(confidence, 4),
                    "Lift": round(lift, 4),
                }
            )

rules_df = (
    pd.DataFrame(rules_list).sort_values("Lift", ascending=False).head(5)
)

print("\n" + "=" * 65)
print("TOP ASSOCIATION RULES BY LIFT (Predicting Congestion)")
print("=" * 65)
print(rules_df.to_string(index=False))


K-MEANS CLUSTERING SUMMARY & PROFILES
          hour  weather_severity  traffic_volume    temp  count dominant_congestion
cluster                                                                            
0        15.06              0.88         3967.95  268.67  10341              Medium
1         2.88              1.97          802.57  279.29  13124                 Low
2        13.55              4.53         4195.23  281.21  11295              Severe
3        15.08              0.68         4328.01  292.72  13444              Severe

TOP ASSOCIATION RULES BY LIFT (Predicting Congestion)
                                Antecedents Consequent (Congestion)  Support  Confidence   Lift
Morning_Peak, Weekday, Weather:Thunderstorm                  Severe   0.0022      0.8974 3.5930
     Morning_Peak, Weekday, Weather:Drizzle                  Severe   0.0047      0.8964 3.5889
         Morning_Peak, Weekday, Weather:Fog                  Severe   0.0038      0.8798 3.5224
        Morning_Pe